# MoneyLens Foundry — Offline Data & Lexicon Distillation Pipeline

**Architecture Principle (Offline-First, Zero APK Bloat):**
This notebook runs on Google Colab GPU (`T4` / `L4` / `A100`) to:
1. Synthesize realistic Malaysian thermal receipt OCR text (5-sen rounding, EN/BM/ZH headers, SST IDs, split-column amounts) and bank/FPX/DuitNow/eWallet notifications using **`vLLM` + JSON-schema Guided/Structured Decoding**.
2. Benchmark and distill **uncovered Malaysian merchant keywords, header noise phrases, and LHDN YA tax-relief rules** using TF-IDF + Logistic Regression residual discovery.
3. Print the distilled `merchant_lexicon_candidates.json` and `evaluation_report.json` directly into the saved notebook output (and optionally mirror to Google Drive) — keeping the Android APK free of `.tflite`/JNI bloat.

> **Compute-Unit Safety:** `try ... finally` flushes output and calls `google.colab.runtime.unassign()` automatically when finished.

In [ ]:
#@title 1. Install vLLM & Prepare Output Directory
import os

os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
MOUNT_GOOGLE_DRIVE = False  #@param {type:"boolean"}

!pip -q install vllm hf_transfer

OUT_DIR = "/content/moneylens_foundry"
if os.path.exists("/content/drive/MyDrive"):
    OUT_DIR = "/content/drive/MyDrive/MoneyLens_Foundry"
elif MOUNT_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        OUT_DIR = "/content/drive/MyDrive/MoneyLens_Foundry"
    except Exception as exc:
        print(f"Drive mount skipped ({exc}); using {OUT_DIR}")

os.makedirs(OUT_DIR, exist_ok=True)
print("Artifacts directory:", OUT_DIR)

In [ ]:
#@title 2. Canonical MoneyLens Taxonomy & Pydantic Schemas (Aligned with format.js & Java Parsers)
from typing import List, Literal, Optional
from pydantic import BaseModel, Field

CANONICAL_TAXONOMY = {
    "Food & drinks": [
        "Hawker & Mamak",
        "Restaurant & Cafe",
        "Fast Food",
        "Food Delivery (GrabFood / Foodpanda)",
        "Coffee, Tea & Drinks",
        "Groceries & Snacks",
    ],
    "Groceries": [
        "Supermarket & Hypermarket",
        "Convenience Store (99 Speedmart / 7-Eleven / FamilyMart)",
        "Wet Market & Fresh Produce",
        "Household & Cleaning Supplies",
    ],
    "Transport": [
        "Toll (TNG / RFID / PayDirect)",
        "Petrol & EV Charging",
        "Parking",
        "Public Transit (MRT / LRT / Bus)",
        "E-Hailing (Grab / AirAsia Ride)",
        "Vehicle Maintenance & Service",
        "Road Tax & Motor Insurance",
        "Car Loan Installment",
    ],
    "Bills & utilities": [
        "Electricity (TNB)",
        "Water & Sewerage (Air Selangor / IWK)",
        "Home Internet & WiFi (Unifi / Time)",
        "Mobile Postpaid & Prepaid",
        "Streaming & App Subscriptions",
        "Condo Maintenance & Sinking Fund",
    ],
    "Loans, BNPL & Credit": [
        "PTPTN Study Loan",
        "BNPL (SPayLater / GrabPayLater / Atome)",
        "Credit Card Bill",
        "Personal Loan",
        "Hire Purchase / Installment",
    ],
    "Housing & property": [
        "Rent",
        "Home Loan / Mortgage",
        "Home Repair & Maintenance",
        "Furniture & Appliances",
        "Cukai Pintu & Cukai Tanah",
    ],
    "Shopping": [
        "Online Marketplace (Shopee / Lazada / TikTok)",
        "Clothing, Bags & Footwear",
        "Electronics & Gadgets",
        "Beauty, Skincare & Grooming",
        "Home & Living",
    ],
    "Health & medical": [
        "Clinic & GP",
        "Pharmacy & Medicine",
        "Hospital & Specialist",
        "Dental & Optical",
        "Full Medical Checkup (LHDN)",
        "Gym, Sports & Fitness",
    ],
    "Insurance & Takaful": [
        "Life Insurance / Takaful",
        "Medical & Health Card",
        "Education Insurance",
        "Personal Accident",
    ],
    "Education": [
        "Higher Education & Tuition",
        "Upskilling & Certification",
        "School & Childcare Fees",
        "Books & Stationery",
    ],
    "Family & gifts": [
        "Parents Allowance",
        "Childcare & Family Support",
        "Festive (Angpao / Duit Raya) & Gifts",
        "Zakat & Fitrah",
        "Charity & Donations",
    ],
    "Entertainment & travel": [
        "Movies, Concerts & Events",
        "Games & Hobbies",
        "Flights & Transport",
        "Hotel & Holiday Stay",
    ],
    "Investments & savings": [
        "EPF Voluntary (i-Saraan)",
        "ASB / Tabung Haji",
        "PRS (Private Retirement Scheme)",
        "SSPN Savings",
        "Stocks, ETF & Crypto",
    ],
    "Taxes & government": [
        "LHDN Income Tax / PCB",
        "JPJ / Renewals / Summons",
        "Government Service & Stamp Duty",
    ],
    "Fees & charges": [
        "Bank & Transfer Fee",
        "Foreign Currency Fee",
        "Late Payment Fee",
        "Card Annual Fee",
    ],
}

CategoryLiteral = Literal[
    "Food & drinks", "Groceries", "Transport", "Bills & utilities",
    "Loans, BNPL & Credit", "Housing & property", "Shopping", "Health & medical",
    "Insurance & Takaful", "Education", "Family & gifts", "Entertainment & travel",
    "Investments & savings", "Taxes & government", "Fees & charges",
]

class SyntheticReceiptCase(BaseModel):
    source_type: Literal["thermal_receipt", "payment_notification"]
    raw_text: str = Field(description="Realistic noisy OCR text or Malaysian banking notification")
    expected_merchant: str = Field(description="Clean brand name without FPX*/iPay88* or Sdn Bhd noise")
    expected_total_myr: str = Field(description="Final net payable MYR amount formatted with 2 decimals, e.g. 12.50")
    expected_date_iso: str = Field(description="ISO date YYYY-MM-DD")
    category: CategoryLiteral
    subcategory: str
    lhdn_bucket: Optional[Literal[
        "Medical & Health", "Sports & Gym", "Education",
        "Lifestyle (Tech / Books / WiFi)", "Life Insurance / Takaful",
        "Medical & Edu Insurance", "SSPN / PRS / EV", "Zakat / Fitrah Rebate",
        "LHDN / PCB Paid", None
    ]] = None

class BatchOutput(BaseModel):
    samples: List[SyntheticReceiptCase]

print(f"Loaded {len(CANONICAL_TAXONOMY)} canonical MoneyLens categories.")

In [ ]:
#@title 3. Generate & Evaluate Malaysian Corpus with vLLM Structured Decoding
#@markdown Selects `Qwen2.5-14B-Instruct-AWQ` on L4/A100 (`sm_80+ >=22GB`), `7B-AWQ` on smaller `sm_80+`, or `Qwen2.5-3B-Instruct` FP16 (`enforce_eager=True`) on T4 (`sm_75`) to avoid quantization/bfloat16 kernel failures.
import gc
import json
import re
import sys
import time
from collections import defaultdict

AUTO_UNASSIGN_RUNTIME = True  #@param {type:"boolean"}
NUM_BATCHES = 10              #@param {type:"integer"}
SAMPLES_PER_BATCH = 6         #@param {type:"integer"}

DATASET_PATH = os.path.join(OUT_DIR, "dataset_malaysian_receipts.jsonl")
LEXICON_PATH = os.path.join(OUT_DIR, "merchant_lexicon_candidates.json")
REPORT_PATH = os.path.join(OUT_DIR, "evaluation_report.json")

JAVA_KNOWN_KEYWORDS = {
    "sspn", "ptptn", "spaylater", "grabpaylater", "atome", "rfid", "paydirect", "toll",
    "petronas", "shell", "petron", "caltex", "bhp", "bhpetrol", "setel", "gentari", "chargev",
    "jomcharge", "dc handal", "shell recharge", "grabfood", "foodpanda", "shopeefood", "beepit",
    "99 speedmart", "7-eleven", "familymart", "kk mart", "mynews", "emart24", "bilabila", "cu mart",
    "lotus", "mydin", "giant", "aeon", "jaya grocer", "village grocer", "econsave", "nsk",
    "hero market", "segi fresh", "mamak", "kopitiam", "pelita", "oriental kopi", "madam kwan",
    "mcdonald", "kfc", "burger king", "marrybrown", "subway", "pizza hut", "domino", "texas chicken",
    "starbucks", "tealive", "zus", "chagee", "mixue", "bask bear", "richiamo", "oldtown",
    "tnb", "tenaga nasional", "air selangor", "indah water", "iwk", "unifi", "time fibre",
    "maxis", "celcom", "digi", "celcomdigi", "umobile", "yes 5g", "netflix", "spotify", "astro",
    "decathlon", "sports direct", "al-ikhsan", "watsons", "guardian", "caring pharmacy", "big pharmacy",
    "alpro", "pathlab", "bp healthcare", "gribbles", "gleneagles", "pantai", "sunway medical", "kpj",
    "columbia asia", "klinik", "clinic", "dental", "pergigian", "focus point", "owndays",
    "prudential", "great eastern", "aia", "allianz", "etiqa", "takaful", "popular", "kinokuniya",
    "bookxcess", "tsutaya", "zakat", "fitrah", "lhdn", "mr diy", "mr. diy", "daiso", "ecoshop",
    "ikea", "machines", "senheng", "harvey norman", "all it", "thundermatch", "uniqlo", "padini",
    "sephora", "gsc", "tgv", "airasia", "malaysia airlines", "agoda", "shopee", "lazada", "zalora",
}

def select_engine_config():
    import torch
    if not torch.cuda.is_available():
        raise RuntimeError("GPU runtime required. Switch Runtime -> Change runtime type -> T4 / L4 / A100.")
    props = torch.cuda.get_device_properties(0)
    vram_gb = props.total_memory / (1024 ** 3)
    cap = (props.major, props.minor)
    print(f"Detected GPU: {props.name} | sm_{props.major}{props.minor} | {vram_gb:.1f} GB VRAM")
    if cap >= (8, 0) and vram_gb >= 22.0:
        return dict(model="Qwen/Qwen2.5-14B-Instruct-AWQ", quantization="awq", dtype="half", enforce_eager=False, max_model_len=4096, gpu_memory_utilization=0.90)
    if cap >= (8, 0) and vram_gb >= 14.0:
        return dict(model="Qwen/Qwen2.5-7B-Instruct-AWQ", quantization="awq", dtype="float16", enforce_eager=True, max_model_len=3072, gpu_memory_utilization=0.88)
    return dict(model="Qwen/Qwen2.5-3B-Instruct", quantization=None, dtype="float16", enforce_eager=True, max_model_len=3072, gpu_memory_utilization=0.85)

def make_structured_sampling(json_schema):
    from vllm import SamplingParams
    try:
        from vllm.sampling_params import StructuredOutputsParams
        return SamplingParams(temperature=0.75, max_tokens=2048, structured_outputs=StructuredOutputsParams(json=json_schema))
    except Exception:
        from vllm.sampling_params import GuidedDecodingParams
        return SamplingParams(temperature=0.75, max_tokens=2048, guided_decoding=GuidedDecodingParams(json=json_schema))

PROMPT_DOMAINS = [
    "Malaysian mamak, kopitiam, cafe, boba & local eateries receipts (including 5-sen rounding & Simplified Tax Invoice headers)",
    "Malaysian neighborhood supermarkets, mini-marts, bakeries & specialty stores (e.g. Jaya Grocer, CS Grocer, TF Value-Mart, Mix.com.my, Chagee, Zus)",
    "DuitNow QR, FPX*, iPay88*, Billplz*, Touch 'n Go eWallet, MAE & Boost notifications with SSM suffixes (Sdn Bhd / Berhad)",
    "LHDN tax-relief merchants: local clinics, dental, optical, diagnostics, EV charging, sports stores, bookstores & tech shops",
    "Malaysian services, utilities, parking apps (FlexiParking, Smart Selangor), tolls, clinics & e-commerce alerts",
]

all_samples = []
llm = None
try:
    from vllm import LLM
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.linear_model import LogisticRegression
    import numpy as np
    import torch

    cfg = select_engine_config()
    print("Loading model config:", cfg)
    llm_kwargs = {k: v for k, v in cfg.items() if v is not None}
    llm = LLM(**llm_kwargs)
    sampling = make_structured_sampling(BatchOutput.model_json_schema())

    taxonomy_str = json.dumps(CANONICAL_TAXONOMY, indent=2)
    prompts = []
    for idx in range(NUM_BATCHES):
        focus = PROMPT_DOMAINS[idx % len(PROMPT_DOMAINS)]
        msg = (
            f"Generate {SAMPLES_PER_BATCH} realistic synthetic Malaysian financial test cases focusing on: {focus}.\n"
            f"Requirements:\n"
            f"- Include diverse real Malaysian merchants beyond major chains (include local kopitiams, clinics, workshops, EV chargers, specialty shops).\n"
            f"- Include OCR noise: address lines, phone numbers, SST IDs, 5-sen rounding ('Rounding Adj +0.02'), split-line 'TOTAL PAYABLE (RM)\\n45.90', gateways ('FPX*', 'IPAY88 *', 'BILLPLZ-'), SSM suffixes ('SDN BHD (123456-X)').\n"
            f"- Use strictly the canonical category and subcategory strings below:\n{taxonomy_str}\n"
            f"- Ensure expected_total_myr is the post-rounding net amount paid.\n"
            f"- Batch #{idx + 1}: do not repeat merchants across items."
        )
        prompts.append(f"<|im_start|>system\nYou are a synthetic Malaysian financial document generator.<|im_end|>\n<|im_start|>user\n{msg}<|im_end|>\n<|im_start|>assistant\n")

    outputs = llm.generate(prompts, sampling)
    for out in outputs:
        raw_json = out.outputs[0].text.strip()
        try:
            parsed = BatchOutput.model_validate_json(raw_json)
            for item in parsed.samples:
                if item.subcategory not in CANONICAL_TAXONOMY.get(item.category, []):
                    continue
                all_samples.append(item.model_dump())
        except Exception as e:
            print("Skipped invalid batch output:", e)

    with open(DATASET_PATH, "w", encoding="utf-8") as f:
        for row in all_samples:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    del llm
    llm = None
    gc.collect()
    torch.cuda.empty_cache()

    uncovered = []
    texts, labels = [], []
    for row in all_samples:
        hay = f"{row['expected_merchant']} {row['raw_text']}".lower()
        matched = any(re.search(rf"(?<![a-z0-9]){re.escape(k)}(?![a-z0-9])", hay) for k in JAVA_KNOWN_KEYWORDS)
        if not matched:
            uncovered.append(row)
        texts.append(f"{row['expected_merchant']} {row['raw_text']}")
        labels.append(f"{row['category']}||{row['subcategory']}")

    lexicon_candidates = defaultdict(list)
    if len(set(labels)) >= 2 and len(texts) >= 15:
        vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1, max_features=3000, stop_words="english")
        X = vec.fit_transform(texts)
        clf = LogisticRegression(max_iter=1000, C=2.0).fit(X, labels)
        vocab = np.array(vec.get_feature_names_out())
        for cls_idx, label in enumerate(clf.classes_):
            weights = clf.coef_[cls_idx]
            top_idx = np.argsort(weights)[-10:][::-1]
            for ti in top_idx:
                tok = str(vocab[ti])
                if weights[ti] > 0.35 and tok not in JAVA_KNOWN_KEYWORDS and not tok.isdigit() and len(tok) >= 3:
                    lexicon_candidates[label].append({"token": tok, "weight": round(float(weights[ti]), 4)})

    with open(LEXICON_PATH, "w", encoding="utf-8") as f:
        json.dump(dict(lexicon_candidates), f, indent=2, ensure_ascii=False)

    report = {
        "model_used": cfg["model"],
        "total_generated": len(all_samples),
        "uncovered_by_current_java_count": len(uncovered),
        "baseline_java_coverage_pct": round(100.0 * (len(all_samples) - len(uncovered)) / max(1, len(all_samples)), 2),
        "uncovered_samples": [{"merchant": u["expected_merchant"], "category": u["category"], "subcategory": u["subcategory"], "source_type": u["source_type"]} for u in uncovered],
        "lexicon_candidates": {k: v[:6] for k, v in lexicon_candidates.items()},
    }
    with open(REPORT_PATH, "w", encoding="utf-8") as f:
        json.dump(report, f, indent=2, ensure_ascii=False)

    print("=== MONEYLENS FOUNDRY RESULT JSON ===")
    print(json.dumps(report, indent=2, ensure_ascii=False))
    print("=== END FOUNDRY RESULT ===")
    sys.stdout.flush()
finally:
    if llm is not None:
        del llm
        gc.collect()
    if AUTO_UNASSIGN_RUNTIME:
        time.sleep(3)
        try:
            from google.colab import runtime
            print("Releasing Colab GPU runtime cleanly...")
            sys.stdout.flush()
            runtime.unassign()
        except Exception:
            pass